In [1]:
import os
import numpy as np
import SimpleITK as sitk
from scipy import ndimage
import nibabel as nib
import os
import threading
import glob
import shutil


In [2]:
data_folder="/home/changh/Documents/2023_DosePredcition/Data/open-kbp-master/nifti_data_test"
os.chdir(data_folder)

In [3]:
class Beam_fdose():
    def __init__(self, ptv_path):
        self.points=[(0,0,0),
                    (40,0,0),
                    (80,0,0),
                    (120,0,0),
                    (160,0,0),
                    (200,0,0),
                    (240,0,0),
                    (280,0,0),
                    (320,0,0)]
        
        self.ptv_path=ptv_path
        self.patient_num=ptv_path.split("/")[-1].split(".")[0].split("_")[0]
        self.binary_ptv=self.get_binary_ptv()
        self.isocenter=self.get_isocenter()
    def get_ptv(self):
        return(sitk.ReadImage(self.ptv_path))
    def get_binary_ptv(self):
        ptv=sitk.GetArrayFromImage(self.get_ptv())
        return np.where(ptv!=0, 1 , 0)
    
    def get_array_ptv(self):
        return  sitk.GetArrayFromImage(self.get_ptv()) 
    def get_array_oar(self):
        splits=self.ptv_path.split("/")
        folder=""
        for i in range(0,len(splits)-1):
            folder=folder=splits[i]
        oar_path=os.path.join(folder,str(self.patient_num)+"_0002.nii.gz"  ) 
        return  sitk.GetArrayFromImage(sitk.ReadImage(oar_path)) 
    def get_array_ct(self):
        splits=self.ptv_path.split("/")
        folder=""
        for i in range(0,len(splits)-1):
            folder=folder=splits[i]
        ct_path=os.path.join(folder,str(self.patient_num)+"_0000.nii.gz"  ) 
        return  sitk.GetArrayFromImage(sitk.ReadImage(ct_path)) 
    def get_isocenter(self):
        #combine=1000*self.get_array_ptv()+0.5*self.get_array_ct()#-1000*self.get_array_oar()
        #print(combine)
     
        array_center=ndimage.center_of_mass(self.get_binary_ptv())
        print(array_center)
        real_center=[]
     
        for i in range(0,3):
            real_center.append(array_center[i]*self.get_ptv().GetSpacing()[i]+self.get_ptv().GetOrigin()[i])
        return real_center
    
    def get_beam_template(self,SAD=1000,field_size=[400,400],beam_value=1): #mm
        # FIXME: change path 
        
        template_path="template/cone_template_"+str(field_size[0]//10)+"_"+str(field_size[1]//10)+"_"+str(SAD//10)+".nii.gz"     
        #euler_transform.SetRotation(0,0,0)
        if not os.path.isfile(template_path):
            print("Creating Beam Template of: field_size=",field_size," ,SAD=",SAD)
            
            cone_boundary=(2*field_size[0],2*field_size[1],2*SAD)
            empty=np.zeros((2*field_size[0],2*SAD,2*field_size[1]),dtype=np.uint8)

            x=cone_boundary[0]
            y=cone_boundary[1]
            z=cone_boundary[2] 
            center=(x/2,y/2)
            for nz in range (0,z):
                #define beam range per slice
                n2x=np.round((x/2)*(nz/z)) # new half x--> x/2:z=n2x:nz
                n2y=np.round((y/2)*(nz/z))
                
                bx=np.round(center[0]-n2x)
                ux=np.round(center[0]+n2x)
                bx=int(np.max((0,bx)))
                ux=int(np.min((x,ux)))

                by=np.round(center[1]-n2y)
                uy=np.round(center[1]+n2y)
                by=int(np.max((0,by)))
                uy=int(np.min((y,uy)))
        
                #fill beam range
                for i in range (bx,ux):
                    for j in range (by,uy):
                        empty[i,nz,j]=beam_value
            
                        
                        
                                
            affine= np.eye(4)
            for i in range(0,2):
                affine[i,i]=-1 
                #
            img = nib.nifti1.Nifti1Image( empty,affine=affine)#np.flip(empty, axis=1),
            nib.save(img,template_path)
            print("Finish!!: field_size=",field_size," ,SAD=",SAD)
        
        img=sitk.ReadImage(template_path)

        return img
    def resample_to_ptv(self,image: sitk.Image,transform=sitk.Euler3DTransform(),
                 interpolator = sitk.sitkNearestNeighbor,default_value=0):

        reference_image=self.get_ptv()
    
        return sitk.Resample(image, reference_image, transform,
                                interpolator, default_value) 
        
    def resample_template_to_iso(self, 
                  image: sitk.Image,
                  origin=None,
                  transform=sitk.Euler3DTransform
                  ):
        
        image.SetOrigin(origin)
        image.SetDirection(self.get_ptv().GetDirection())

        resample = sitk.ResampleImageFilter()
        #resample.SetTransform(transform)
        resample.SetReferenceImage(image)
        resample.SetInterpolator(sitk.sitkNearestNeighbor)
    
        return resample.Execute(image)
    
    def get_beam(self,beams=None,field_size=[400,400],SAD=1000,dtype=sitk.sitkUInt64): #  each plan #beams:list[Beam]
        resamples=None

              
        cone=self.get_beam_template(field_size=field_size,SAD=SAD)

        #FIXME: if SAD and field size didn't match template, rescale
        #FIXME: if beam value is different, amplify
        
        center=(field_size[0]-1,SAD-1,field_size[1]-1)
        temp_origin=[(self.isocenter[i]-center[i]) for i in range(0,3)]  
        cone=self.resample_template_to_iso(image=cone, origin=temp_origin) 
        transform=sitk.Euler3DTransform()

        cone=self.resample_to_ptv(cone,transform)
        #sitk.WriteImage(cone,os.path.join(data_folder,"beam","cone_align"+".nii.gz"))


        
        for i in range(0,len(self.points)):
            theta=self.points[i]
            #print(theta)
            theta=[np.deg2rad(theta[i]) for i in range(0,3)]
            
            chain_cone=cone
            for r in range(0,3):#[1,0,2]
                r_theta=[0,0,0]
                if theta[r]!=0:
                    r_theta[r]=theta[r]
                    
                    transform=sitk.Euler3DTransform()
                    transform.SetCenter(self.isocenter)
                    transform.SetRotation(r_theta[0],r_theta[1],r_theta[2])#NOTE
                    chain_cone=self.resample_to_ptv(chain_cone,transform)

            if resamples==None: 
                resamples =sitk.Cast(chain_cone,dtype)
            else:
                resamples= resamples+ sitk.Cast(chain_cone,dtype)

              
        sitk.WriteImage(resamples,os.path.join(data_folder,"beam",self.patient_num+"_0003.nii.gz"))

        
        

In [4]:
import glob
import os
files=sorted(glob.glob(os.path.join("imagesTs","*_0001.nii.gz")))

In [5]:
for file in files:
    beam=Beam_fdose(file)
    beam.get_beam(dtype=sitk.sitkUInt8,field_size=[50,50])
   

(60.06787027749916, 63.38214643931796, 61.053744567034435)
(62.06059329175796, 64.12329344345189, 67.0903421540536)
(61.601279874624524, 57.90087501632493, 62.86117278307431)
(61.461565668474414, 63.61801461919359, 65.6420655505777)
(59.572169059011166, 63.588250930356196, 64.87409179514442)
(58.95126686246204, 65.4854397929215, 58.6090397730101)
(60.53676321652826, 63.9782256431031, 62.507291877658496)
(60.61558731123949, 61.72017837235229, 64.71622580318233)
(58.59844159851875, 61.98403024224657, 65.42771177287456)
(60.58370837083709, 63.44109410941094, 66.43604360436044)
(59.65267925818306, 62.33356123408256, 61.33022818562516)
(59.01137342753748, 65.6219196967086, 66.05971049457177)
(60.26477883422902, 65.11244315832988, 65.73770152955767)
(57.81351351351351, 65.18432432432432, 60.84262548262548)
(54.57152569259532, 64.04390686080109, 59.47328763242876)
(58.71606080634501, 64.18783873099802, 62.529676140118966)
(57.12245823998185, 62.97546864808145, 60.61444655568475)
(60.413185051

In [6]:
beam.isocenter

[238.9547329461517, 257.4897672490633, 167.1959972394755]